In [1]:
import pickle
import pandas as pd

# Load models
bat_model = pickle.load(open("rf_bat.pkl", "rb"))
bowl_model = pickle.load(open("ridge_bowl.pkl", "rb"))

# Load feature columns
bat_features = pickle.load(open("bat_features.pkl", "rb"))
bowl_features = pickle.load(open("bowl_features.pkl", "rb"))

print("Models and features loaded successfully")

Models and features loaded successfully


In [2]:
import pandas as pd

batting = pd.read_csv("batting_clean.csv")

# recreate season_year if needed
if "season_year" not in batting.columns:
    batting["season_year"] = batting["season"].astype(str).str[:4].astype(int)

In [3]:
# Create sample input (batting)
sample_bat = pd.DataFrame(columns=bat_features)

sample_bat.loc[0] = 0  # initialize
sample_bat["player"] = "V Kohli"
sample_bat["balls"] = 30
sample_bat["strike_rate"] = 140
sample_bat["low_score"] = 0
sample_bat["season"] = "2024"
sample_bat["venue"] = "Wankhede Stadium"
sample_bat["match_type"] = "T20"
sample_bat["stage_bucket"] = "league"
sample_bat["match_year"] = 2024
sample_bat["match_month"] = 4
sample_bat["match_weekday"] = 2
sample_bat["season_year"] = 2024

# Predict
prediction = bat_model.predict(sample_bat)[0]
print("Predicted runs:", prediction)

Predicted runs: 46.348852626691226


In [4]:
import pickle

bowl_model = pickle.load(open("ridge_bowl.pkl", "rb"))
bowl_features = pickle.load(open("bowl_features.pkl", "rb"))

In [5]:
# Create sample input (bowling)
sample_bowl = pd.DataFrame(columns=bowl_features)

sample_bowl.loc[0] = 0  # initialize

sample_bowl["bowler"] = "J Bumrah"
sample_bowl["balls"] = 24
sample_bowl["runs_conceded"] = 30
sample_bowl["wides"] = 2
sample_bowl["no_balls"] = 0
sample_bowl["wickets"] = 1
sample_bowl["dot_balls"] = 10

# derived features
sample_bowl["overs"] = sample_bowl["balls"] / 6
sample_bowl["strike_rate_balls_per_wicket"] = sample_bowl["balls"] / sample_bowl["wickets"]
sample_bowl["dot_ball_rate"] = sample_bowl["dot_balls"] / sample_bowl["balls"]

# categorical
sample_bowl["season"] = "2024"
sample_bowl["venue"] = "Wankhede Stadium, Mumbai"
sample_bowl["match_type"] = "League"
sample_bowl["stage_bucket"] = "league"

# time features
sample_bowl["season_year"] = 2024
sample_bowl["match_year"] = 2024
sample_bowl["match_month"] = 4
sample_bowl["match_weekday"] = 2

# match_id (important!)
sample_bowl["match_id"] = 0

# Predict
prediction = bowl_model.predict(sample_bowl)[0]
print("Predicted economy:", prediction)

Predicted economy: 7.4941720597683315


In [6]:
import pickle
import pandas as pd
import numpy as np

# Load saved batting model + features
bat_model = pickle.load(open("rf_bat.pkl", "rb"))
bat_features = pickle.load(open("bat_features.pkl", "rb"))

# Load data
batting = pd.read_csv("batting_clean.csv")

# Recreate season_year if needed
if "season_year" not in batting.columns:
    batting["season_year"] = batting["season"].astype(str).str[:4].astype(int)

In [7]:
def predict_existing_batting(player_name, stage, season, venue, match_type):
    data = batting[
        (batting["player"] == player_name) &
        (batting["season_year"] < int(season))
    ].copy()

    if data.empty:
        print(f"No history found for {player_name} before {season}.")
        return None

    row = pd.DataFrame(columns=bat_features, index=[0])

    for col in bat_features:
        if col in data.columns:
            if pd.api.types.is_numeric_dtype(data[col]):
                row.loc[0, col] = data[col].mean()
            else:
                mode_val = data[col].mode()
                row.loc[0, col] = mode_val.iloc[0] if not mode_val.empty else np.nan

    row.loc[0, "player"] = player_name
    row.loc[0, "stage_bucket"] = stage
    row.loc[0, "season"] = str(season)
    row.loc[0, "match_year"] = int(season)
    row.loc[0, "season_year"] = int(season)
    row.loc[0, "match_id"] = 0
    row.loc[0, "venue"] = venue
    row.loc[0, "match_type"] = match_type

    overall_avg = data["runs"].mean()
    venue_data = data[data["venue"] == venue]
    stage_data = data[data["stage_bucket"] == stage]

    venue_diff = (venue_data["runs"].mean() - overall_avg) if len(venue_data) >= 3 else 0
    venue_weight = min(len(venue_data) / 20, 0.4) if len(venue_data) >= 3 else 0

    stage_diff = (stage_data["runs"].mean() - overall_avg) if len(stage_data) >= 3 else 0
    stage_weight = min(len(stage_data) / 20, 0.3) if len(stage_data) >= 3 else 0

    venue_adjustment = venue_diff * venue_weight
    stage_adjustment = stage_diff * stage_weight

    base = bat_model.predict(row)[0]
    final = max(0, base + venue_adjustment + stage_adjustment)

    print("Player:", player_name)
    print("Overall avg:", overall_avg)
    print("Venue avg:", venue_data["runs"].mean() if len(venue_data) > 0 else None)
    print("Stage avg:", stage_data["runs"].mean() if len(stage_data) > 0 else None)
    print("Venue count:", len(venue_data))
    print("Stage count:", len(stage_data))
    print("Venue adjustment:", venue_adjustment)
    print("Stage adjustment:", stage_adjustment)
    print("Base prediction:", base)
    print("Final prediction:", final)

    return final

In [8]:
predict_existing_batting(
    player_name="V Kohli",
    stage="league",
    season="2024",
    venue="Wankhede Stadium, Mumbai",
    match_type="League"
)

Player: V Kohli
Overall avg: 31.759825327510917
Venue avg: 24.428571428571427
Stage avg: 32.395348837209305
Venue count: 7
Stage count: 215
Venue adjustment: -2.5659388646288215
Stage adjustment: 0.19065705290951626
Base prediction: 32.12705819525876
Final prediction: 29.751776383539454


np.float64(29.751776383539454)

In [11]:
import pandas as pd

bowling = pd.read_csv("bowling_clean.csv")

# recreate season_year if missing
if "season_year" not in bowling.columns:
    bowling["season_year"] = bowling["season"].astype(str).str[:4].astype(int)

print("Bowling data loaded successfully")

Bowling data loaded successfully


In [12]:
def predict_existing_bowling(player_name, stage, season, venue, match_type):
    data = bowling[
        (bowling["bowler"] == player_name) &
        (bowling["season_year"] < int(season))
    ].copy()

    if data.empty:
        print(f"No history found for {player_name} before {season}.")
        return None

    row = pd.DataFrame(columns=bowl_features, index=[0])

    for col in bowl_features:
        if col in data.columns:
            if pd.api.types.is_numeric_dtype(data[col]):
                row.loc[0, col] = data[col].mean()
            else:
                mode_val = data[col].mode()
                row.loc[0, col] = mode_val.iloc[0] if not mode_val.empty else np.nan

    row.loc[0, "bowler"] = player_name
    row.loc[0, "stage_bucket"] = stage
    row.loc[0, "season"] = str(season)
    row.loc[0, "match_year"] = int(season)
    row.loc[0, "season_year"] = int(season)
    row.loc[0, "match_id"] = 0
    row.loc[0, "venue"] = venue
    row.loc[0, "match_type"] = match_type

    overall_avg = data["economy"].mean()
    venue_data = data[data["venue"] == venue]
    stage_data = data[data["stage_bucket"] == stage]

    venue_diff = (venue_data["economy"].mean() - overall_avg) if len(venue_data) >= 3 else 0
    venue_weight = min(len(venue_data) / 20, 0.4) if len(venue_data) >= 3 else 0

    stage_diff = (stage_data["economy"].mean() - overall_avg) if len(stage_data) >= 3 else 0
    stage_weight = min(len(stage_data) / 20, 0.3) if len(stage_data) >= 3 else 0

    venue_adjustment = venue_diff * venue_weight
    stage_adjustment = stage_diff * stage_weight

    base = bowl_model.predict(row)[0]
    final = max(0, base + venue_adjustment + stage_adjustment)

    print("Bowler:", player_name)
    print("Overall avg economy:", overall_avg)
    print("Venue avg economy:", venue_data["economy"].mean() if len(venue_data) > 0 else None)
    print("Stage avg economy:", stage_data["economy"].mean() if len(stage_data) > 0 else None)
    print("Venue count:", len(venue_data))
    print("Stage count:", len(stage_data))
    print("Venue adjustment:", venue_adjustment)
    print("Stage adjustment:", stage_adjustment)
    print("Base prediction:", base)
    print("Final prediction:", final)

    return final

In [13]:
predict_existing_bowling(
    player_name="JJ Bumrah",
    stage="league",
    season="2024",
    venue="Wankhede Stadium, Mumbai",
    match_type="League"
)

Bowler: JJ Bumrah
Overall avg economy: 7.313608961889054
Venue avg economy: 6.5
Stage avg economy: 7.382276423970156
Venue count: 4
Stage count: 112
Venue adjustment: -0.16272179237781081
Stage adjustment: 0.020600238624330556
Base prediction: 7.217757534911245
Final prediction: 7.075635981157765


np.float64(7.075635981157765)